In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import when, col, from_unixtime, to_date
from pyspark.sql.functions import sum as _sum

df_games = spark.read.format("delta").load("dbfs:/Volumes/workspace/steam/steam_volume/silver")
df_prices = spark.read.format("delta").load("dbfs:/Volumes/workspace/steam/steam_volume/silver_prices")

df_games_with_date = df_games.withColumn(
    "last_played_date",
    when(col("last_played_unix") == 0, None)
    .otherwise(to_date(from_unixtime(col("last_played_unix"))))
)

df_gold = (
    df_games_with_date.alias("g")
    .join(df_prices.alias("p"), col("g.appid") == col("p.appid"), "left")
    .select(
        col("g.*"),
        col("p.price_final_formatted"),
        col("p.price_initial_formatted"),
        col("p.discount_percent"),
        col("p.price_final_cents"),
        col("p.price_initial_cents"),
        col("p.currency"),
        col("p.is_free").alias("is_free_game")
    )
)


gold_path = "dbfs:/Volumes/workspace/steam/steam_volume/gold"

df_gold.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(gold_path)



In [0]:
%sql
CREATE OR REPLACE TABLE workspace.steam.gold_df AS
SELECT *
FROM delta.`/Volumes/workspace/steam/steam_volume/gold`;
